# Turkish-CBE P1 — Colab A100 leg (qwen2.5 3b / 7b / 14b)

Runs the frozen 16,618-prompt manifest on the remaining models with the exact
code and model digests used on the M1. Results stream to Google Drive, so
disconnects are harmless: re-running the run cell resumes exactly (completed
prompt IDs are skipped; duplicates impossible).

**Before running:** upload `p1_colab_bundle.zip` to `MyDrive/turkish-cbe-p1/`.
Runtime: GPU (A100). To pause gracefully: create an empty file named `PAUSE`
in `MyDrive/turkish-cbe-p1/results/` (takes effect within one call); delete it
and re-run the run cell to resume.

In [ ]:
# 1 — environment check
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout)

In [ ]:
# 2 — mount Drive, unpack the verified bundle
from google.colab import drive
drive.mount('/content/drive')
import hashlib, zipfile, pathlib, os
WORK = pathlib.Path('/content/drive/MyDrive/turkish-cbe-p1')
BUNDLE = WORK / 'p1_colab_bundle.zip'
EXPECTED_ZIP_SHA = '97ba8387a632031643623ef49497d165b5ec0b6a8e741280a7bddfbf45a185c7'
EXPECTED_MANIFEST_SHA = '34f76f726e75f954e8326a491953ad5486b3cec51be3ad95a3aeb5debdd1242e'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == EXPECTED_ZIP_SHA, 'bundle hash mismatch'
os.makedirs('/content/p1', exist_ok=True)
zipfile.ZipFile(BUNDLE).extractall('/content/p1')
manifest = pathlib.Path('/content/p1/data/manifests/p1_expanded_manifest.jsonl')
assert hashlib.sha256(manifest.read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA, 'manifest hash mismatch'
(WORK / 'results').mkdir(exist_ok=True)
print('bundle + manifest verified; results dir:', WORK / 'results')

In [ ]:
# 3 — install Ollama and start the server (models on fast local disk)
import subprocess, time, urllib.request, json, os
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
env = dict(os.environ, OLLAMA_MODELS='/content/ollama_models')
server = subprocess.Popen(['ollama','serve'], env=env,
                          stdout=open('/content/ollama.log','w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(json.load(urllib.request.urlopen('http://localhost:11434/api/version'))); break
    except Exception: time.sleep(1)

In [ ]:
# 4 — pull models and verify digests equal the M1 run's digests
import subprocess, urllib.request, json
EXPECTED = {
 'qwen2.5:3b':  '357c53fb659c5076de1d65ccb0b397446227b71a42be9d1603d46168015c9e4b',
 'qwen2.5:7b':  '845dbda0ea48ed749caafd9e6037047aa19acfcfd82e704d7ca97d631a0b697e',
 'qwen2.5:14b': '7cdf5a0187d5c58cc5d369b255592f7841d1c4696d45a8c8a9489440385b22f6',
}
for model in EXPECTED:
    subprocess.run(['ollama','pull',model], check=True)
tags = {m['name']: m['digest'] for m in
        json.load(urllib.request.urlopen('http://localhost:11434/api/tags'))['models']}
for model, digest in EXPECTED.items():
    assert tags.get(model) == digest, f'{model}: digest {tags.get(model)} != M1 {digest}'
print('all three digests identical to the M1 arm')

In [ ]:
# 5 — RUN (re-run this cell any time to resume; PAUSE file pauses gracefully)
import subprocess, pathlib, sys
WORK = pathlib.Path('/content/drive/MyDrive/turkish-cbe-p1')
PAUSE = WORK / 'results' / 'PAUSE'
STATUS = WORK / 'results' / 'RUN_STATUS_colab.txt'
for model in ('qwen2.5:3b','qwen2.5:7b','qwen2.5:14b'):
    safe = model.replace(':','-').replace('.','-')
    out = WORK / 'results' / f'{safe}.jsonl'
    with STATUS.open('a') as s: s.write(f'start {model}\n')
    code = subprocess.run([sys.executable,'-m','inference.ollama_runner',
        '--manifest','data/manifests/p1_expanded_manifest.jsonl',
        '--model',model,'--output',str(out),
        '--progress-every','200','--pause-file',str(PAUSE)],
        cwd='/content/p1', env={'PYTHONPATH':'/content/p1/src','PATH':'/usr/local/bin:/usr/bin:/bin'},
    ).returncode
    with STATUS.open('a') as s: s.write(f'exit {model} code={code}\n')
    if code == 75:
        print('paused by PAUSE flag; delete it and re-run this cell to resume'); break
    if code != 0:
        print(f'{model} exited {code}; re-run this cell to resume'); break
else:
    print('ALL THREE MODELS COMPLETE')

In [ ]:
# 6 — progress / integrity counts
import pathlib, json
WORK = pathlib.Path('/content/drive/MyDrive/turkish-cbe-p1')
for f in sorted((WORK/'results').glob('qwen2-5-*.jsonl')):
    n = sum(1 for l in f.open() if '"record_kind": "inference"' in l)
    print(f.name, n, '/ 16618')